# TDAH and MI result generation

Set `DEBUG = True` in the setup cell to write CSV outputs under `Temp/Results`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/STF-KernelSHAP.git"
REPO_NAME = "STF-KernelSHAP"
PACKAGE_PATH = Path("src") / "stf_kernelshap"


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / PACKAGE_PATH).exists():
            return candidate

    working_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else cwd
    clone_dir = working_root / REPO_NAME
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        check=True,
    )


PROJECT_ROOT = resolve_project_root()
os.chdir(PROJECT_ROOT)

RUNNING_IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
RUNNING_IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").exists()
INSTALL_REQUIREMENTS = RUNNING_IN_COLAB or RUNNING_IN_KAGGLE

if INSTALL_REQUIREMENTS:
    install_project_requirements(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INSTALL_REQUIREMENTS:", INSTALL_REQUIREMENTS)


In [ ]:
from stf_kernelshap.notebook_setup import setup_notebook_environment

DEBUG = True
paths = setup_notebook_environment(debug=DEBUG)

DATA_DIR = paths.data_dir
MODELS_DIR = paths.models_dir
OUTPUT_MODELS_DIR = paths.output_models_dir
RESULTS_DIR = paths.results_dir
FIGURES_DIR = paths.figures_dir


In [ ]:
import pickle

from stf_kernelshap.evaluation.pipeline import (
    evaluate_all_windows_to_single_csv,
    train_tdah_with_best_optuna_config,
)
from stf_kernelshap.data import get_segmented_data


In [ ]:
model_name = "shallowconvnet"  # "eegnet", "shallowconvnet", "tgarnet"
folds_path = DATA_DIR / "TDAH" / "folds.pkl"
journal_file = MODELS_DIR / "TDAH" / "Optuna" / f"study_{model_name}_TDAH.journal"
study_name = f"study_{model_name}_TDAH"

base_model_args = {
    "nb_classes": 2,
    "Chans": 19,
    "Samples": 512,
}


In [ ]:
with open(folds_path, "rb") as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data(
    path_adhd=str(DATA_DIR / "TDAH" / "ieee" / "ADHD_group"),
    path_control=str(DATA_DIR / "TDAH" / "ieee" / "Control_group"),
)


In [ ]:
results = train_tdah_with_best_optuna_config(
    model_name=model_name,
    study_name=study_name,
    journal_file=str(journal_file),
    base_model_args=base_model_args,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    epochs=2,
    batch_size=16,
    seed=34,
    seed_gap=100,
    n_repeats=2,
)

df_predictions = results["subject_predictions_df"]
tdah_output_csv = RESULTS_DIR / "TDAH_results.csv"
df_predictions.to_csv(tdah_output_csv, index=False)
df_predictions


In [ ]:
df_results = evaluate_all_windows_to_single_csv(
    data_mi_root=str(DATA_DIR / "MI"),
    models_mi_root=str(MODELS_DIR / "MI"),
    output_csv=str(RESULTS_DIR / "MI_results.csv"),
)
df_results
